In [ ]:
from dotenv import load_dotenv

# Load environment variables from .env
load_dotenv()

In [ ]:
from typing import TypedDict

from langchain.agents.middleware import dynamic_prompt
from langgraph.checkpoint.memory import InMemorySaver


# Runtime context schema
class RuntimeContext(TypedDict):
    user_id: str
    writing_goal: str
    tone: str


# Dynamic system prompt that uses runtime context per request
@dynamic_prompt
def persona_prompt(request: object) -> str:
    ctx: RuntimeContext = request.runtime.context
    return (
        "You are an assistant for Inkwell, a markdown writing studio for developer-writers.\n"
        f"User: {ctx['user_id']}\n"
        f"Writing goal: {ctx['writing_goal']}\n"
        f"Preferred tone: {ctx['tone']}\n"
        "Give practical, short, implementation-ready guidance."
    )


# In-memory checkpointing for multi-turn continuity
memory = InMemorySaver()

In [ ]:
from langchain.agents import create_agent
from langchain_core.messages import HumanMessage

contextual_agent = create_agent(
    model="claude-haiku-4-5",
    middleware=[persona_prompt],
    context_schema=RuntimeContext,
    checkpointer=memory,
)

runtime_context: RuntimeContext = {
    "user_id": "linnie",
    "writing_goal": "ship an article editor with FastAPI + React",
    "tone": "concise",
}

thread_cfg = {"configurable": {"thread_id": "inkwell-session-1"}}

In [ ]:
# Turn 1
r1 = contextual_agent.invoke(
    {"messages": [HumanMessage(content="Draft a 3-step plan to add autosave to Inkwell.")]},
    context=runtime_context,
    config=thread_cfg,
)
print("Turn 1:", r1["messages"][-1].content)

In [ ]:
# Turn 2 (same thread_id => state restored via InMemorySaver)
r2 = contextual_agent.invoke(
    {
        "messages": [
            HumanMessage(content="Now turn that into a checklist with acceptance criteria.")
        ]
    },
    context=runtime_context,
    config=thread_cfg,
)
print("Turn 2:", r2["messages"][-1].content)